# 03. FAISS 벡터 검색 시스템 — 실험 1 모델 기반

## 실험 개요

이 노트북은 실험 1(`best_model.pth`, Val Loss 0.005174)로 학습된 ResNet18 모델을 사용해 49,987개 패턴의 512차원 임베딩을 추출하고, FAISS L2 인덱스로 유사 패턴 검색 시스템을 구축한다.

검색은 Top-100 후보를 뽑은 뒤 시간 다양성 필터링을 적용해 최종 Top-3를 선택하는 방식으로 동작한다.

## 작업 단계

| 단계 | 설명 | 예상 시간 |
|---|---|---|
| 1. 모델 로드 | Google Drive에서 best_model.pth 로드 | 1분 |
| 2. 이미지 압축 해제 | images.tar → 로컬 | 5~10분 |
| 3. 임베딩 생성 | 49,987개 이미지 → 512차원 벡터 | 30~60분 |
| 4. FAISS 인덱스 구축 | L2 IndexFlatL2 | 1분 미만 |
| 5. 검색 테스트 | 시각화 및 성능 평가 | 5분 |

## 저장 결과

- `embeddings.npy` — (49987, 512) float32 행렬
- `image_paths.pkl` — 이미지 경로 매핑
- `faiss_index.bin` — FAISS 인덱스 (FastAPI에서 직접 로드)
- `search_results.png` — 검색 결과 시각화

## 1. 환경 설정 및 라이브러리

FAISS CPU 버전을 설치하고 PyTorch, numpy, matplotlib 등 필요한 라이브러리를 불러온다.

FAISS는 Facebook AI Research에서 개발한 벡터 유사도 검색 라이브러리로, 50,000개 512차원 벡터에서 가장 가까운 이웃을 밀리초 단위로 찾는다. CPU 버전으로도 이 규모에서는 충분히 빠르다.

GPU 사용 가능 여부를 확인한다. 임베딩 추출은 GPU가 있어야 30~60분 안에 완료되므로, 시작 전 반드시 CUDA 환경을 점검한다.

In [ ]:
# GPU 확인
!nvidia-smi

FAISS CPU 버전을 설치한다. 이 노트북 규모(50k 벡터)에서는 CPU 버전으로도 검색 latency가 충분히 낮다.

In [ ]:
# Faiss 설치 (CPU 버전)
# patron_05는 테스트용이므로 CPU 모드로 충분합니다
!pip install faiss-cpu -q

PyTorch, FAISS, numpy, matplotlib 등 필요한 라이브러리를 불러온다. PyTorch 버전과 GPU 이름을 출력해 환경을 최종 확인한다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import faiss
import pickle
import os
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from google.colab import drive

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Faiss Version: {faiss.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Google Drive 마운트 및 데이터 준비

Google Drive를 마운트하고 모델·이미지·출력 경로를 설정한다.

임베딩 생성에 필요한 이미지는 Google Drive에서 Colab 로컬로 복사해 사용한다. Google Drive I/O는 로컬 대비 약 100배 느리므로, 임베딩 추출 루프는 반드시 로컬 저장소에서 읽어야 한다.

In [ ]:
# Google Drive 마운트
drive.mount('/content/drive')

# 경로 설정
MODEL_PATH = '/content/drive/MyDrive/Patron/models/best_model.pth'
DATA_DIR = Path('/content/data/images')
OUTPUT_DIR = Path('/content/drive/MyDrive/Patron/faiss')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ 모델 경로: {MODEL_PATH}")
print(f"✅ 데이터 경로: {DATA_DIR}")
print(f"✅ 출력 경로: {OUTPUT_DIR}")

`images.tar`를 Google Drive에서 Colab 로컬로 복사하고 압축을 해제한다. 메타데이터 CSV도 로드해 패턴 수와 컬럼 목록을 확인한다.

In [ ]:
# images.tar 업로드 및 압축 해제
# 📌 Google Drive에서 images.tar을 Colab으로 복사
!mkdir -p /content/data
!cp /content/drive/MyDrive/Patron/images.tar /content/data/

# 압축 해제 (5-10분 소요)
!tar -xf /content/data/images.tar -C /content/data/

# .npy 파일 개수 확인 (patron_03에서 npy로 저장됨)
image_files = sorted(list(DATA_DIR.glob('*.npy')))
print(f"\n✅ 총 이미지 개수: {len(image_files):,}개")
print(f"✅ 첫 번째 이미지: {image_files[0].name}")
print(f"✅ 마지막 이미지: {image_files[-1].name}")

# 메타데이터 로드 (patron_03에서 생성한 경로)
import pandas as pd
METADATA_PATH = '/content/drive/MyDrive/Patron/data/processed/metadata_all.csv'
metadata_df = pd.read_csv(METADATA_PATH)
print(f"\n✅ 메타데이터 로드 완료: {len(metadata_df):,}개 패턴")
print(f"✅ 컬럼: {list(metadata_df.columns)}")

## 3. ResNet18 모델 로드

학습된 ResNet18 모델 아키텍처를 정의하고 Google Drive에서 체크포인트를 로드한다.

학습 당시와 동일한 구조(1채널 입력, 512차원 L2 정규화)를 사용해야 임베딩이 일치한다. 체크포인트에서 `best_epoch`와 `val_loss`를 확인해 올바른 모델을 로드했는지 검증한다.

ResNet18 모델을 정의하고 체크포인트를 로드한다. best_model.pth는 Val Loss 0.005174 (Epoch 2)로 학습된 최적 모델이다.

In [ ]:
# ResNet18 모델 정의 (patron_03과 동일한 구조)
class ChartEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512):
        super(ChartEmbeddingModel, self).__init__()
        
        resnet = models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.embedding_dim = embedding_dim
    
    def forward(self, x):
        features = self.features(x)
        embeddings = features.view(features.size(0), -1)  # 512차원
        embeddings = nn.functional.normalize(embeddings, p=2, dim=1)  # L2 정규화
        return embeddings

# 모델 로드
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChartEmbeddingModel(embedding_dim=512).to(device)

# 체크포인트 로드
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ 모델 로드 완료!")
print(f"✅ Best Epoch: {checkpoint['epoch']}")
print(f"✅ Best Val Loss: {checkpoint['val_loss']:.6f}")
print(f"✅ Device: {device}")

## 4. 데이터셋 및 DataLoader 준비

ImageDataset 클래스를 정의하고 배치 크기 256의 DataLoader를 생성한다.

`.npy` 파일을 로드해 PIL 이미지로 변환하고, `pin_memory=True`로 CPU→GPU 전송 속도를 높인다. 4개 워커로 데이터 로딩을 병렬화해 GPU 유휴 시간을 줄인다.

ImageDataset 클래스와 DataLoader를 정의한다. 배치 크기 256은 A100 40GB VRAM에서 처리 효율이 높은 설정이다.

In [ ]:
# 이미지 변환
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# 데이터셋 정의 (.npy 파일용)
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # .npy 파일 로드 (Shape: (12, 4) - 12주 x 4가격)
        data = np.load(img_path)
        
        # 0~255 범위로 변환 후 PIL Image로 변환
        data_normalized = (data * 255).astype(np.uint8)
        image = Image.fromarray(data_normalized, mode='L')
        
        if self.transform:
            image = self.transform(image)
        
        return image, str(img_path)

# 데이터셋 및 DataLoader 생성
dataset = ImageDataset(image_files, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=256,  # A100에 맞춰 큰 배치 사용
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"✅ 데이터셋 크기: {len(dataset):,}")
print(f"✅ 배치 개수: {len(dataloader):,}")
print(f"✅ 배치 크기: 256")

## 5. 임베딩 생성 (30~60분)

전체 이미지를 배치 단위로 모델에 통과시켜 512차원 L2 정규화 임베딩을 추출한다.

`torch.no_grad()` 모드로 실행해 gradient 저장을 생략하고 메모리를 절약한다. 배치별 결과를 리스트에 누적하고 마지막에 `np.vstack()`으로 (N, 512) float32 행렬로 합친다.

배치 단위로 임베딩을 추출한다. 완료 후 (49987, 512) 행렬이 생성된다.

In [ ]:
# 임베딩 추출
all_embeddings = []
all_paths = []

model.eval()
with torch.no_grad():
    for images, paths in tqdm(dataloader, desc="임베딩 생성 중"):
        images = images.to(device)
        
        # 임베딩 추출
        embeddings = model(images)
        
        # CPU로 이동 및 저장
        all_embeddings.append(embeddings.cpu().numpy())
        all_paths.extend(paths)

# Numpy 배열로 변환
embeddings_matrix = np.vstack(all_embeddings).astype('float32')

print(f"\n✅ 임베딩 생성 완료!")
print(f"✅ Shape: {embeddings_matrix.shape}")
print(f"✅ 총 이미지: {len(all_paths):,}개")
print(f"✅ 임베딩 차원: {embeddings_matrix.shape[1]}차원")

임베딩 행렬을 `embeddings.npy`로, 이미지 경로 목록을 `image_paths.pkl`로 Google Drive에 저장한다. 이 파일들이 있으면 다음 세션에서 임베딩 생성 단계를 건너뛸 수 있다.

In [ ]:
# 임베딩 저장
np.save(OUTPUT_DIR / 'embeddings.npy', embeddings_matrix)
with open(OUTPUT_DIR / 'image_paths.pkl', 'wb') as f:
    pickle.dump(all_paths, f)

print(f"✅ 임베딩 저장 완료: {OUTPUT_DIR / 'embeddings.npy'}")
print(f"✅ 경로 저장 완료: {OUTPUT_DIR / 'image_paths.pkl'}")

## 6. FAISS 인덱스 구축 및 저장

512차원 L2 IndexFlatL2 인덱스를 만들고 전체 임베딩을 추가한 뒤 `faiss_index.bin`으로 저장한다.

IndexFlatL2는 완전 탐색(brute force) 방식으로 정확한 최근접 이웃을 반환한다. 50k 규모에서는 근사 인덱스(IVF, HNSW)보다 단순한 Flat 인덱스로도 충분히 빠르다. FastAPI 서비스가 시작할 때 이 파일을 로드해 인덱스 재구축 시간을 절약한다.

FAISS L2 인덱스를 구축한다. 총 49,987개 벡터가 추가되며 벡터 차원은 512다.

In [ ]:
# Faiss 인덱스 생성 (L2 거리 기반)
embedding_dim = embeddings_matrix.shape[1]
index = faiss.IndexFlatL2(embedding_dim)

# CPU 모드 사용 (patron_05는 테스트용이므로 충분히 빠름)
print("✅ Faiss CPU 모드 사용")

# 임베딩 추가
index.add(embeddings_matrix)

print(f"\n✅ Faiss 인덱스 구축 완료!")
print(f"✅ 총 벡터 개수: {index.ntotal:,}")
print(f"✅ 벡터 차원: {embedding_dim}")

구축된 FAISS 인덱스를 `faiss_index.bin`으로 저장합니다. FastAPI 서비스가 시작할 때 이 파일을 로드하면 인덱스를 매번 새로 만들 필요가 없어 시작 시간이 크게 줄어듭니다.

In [ ]:
# Faiss 인덱스 저장
faiss.write_index(index, str(OUTPUT_DIR / 'faiss_index.bin'))
print(f"✅ Faiss 인덱스 저장 완료: {OUTPUT_DIR / 'faiss_index.bin'}")

## 7. 검색 테스트 및 시각화

시간 다양성 필터링 함수와 통합 검색 함수를 정의하고 랜덤 쿼리로 테스트한다.

시간 다양성 필터링 이유: 동일 종목의 인접한 시기 패턴은 유사하기 때문에 Top-10을 그냥 반환하면 같은 종목·같은 기간 패턴이 몰릴 수 있다. Top-100 후보를 뽑고 날짜가 서로 최대한 멀리 떨어진 3개를 선택해 다양한 시장 상황의 유사 패턴을 보여준다.

시간 다양성 필터링 함수와 검색 함수를 정의하고 단일 쿼리로 실행한다. Top-1은 가장 유사한 패턴, Top-2는 Top-1과 날짜 차이 최대, Top-3는 Top-1·2와 모두 날짜 차이가 최대인 패턴을 선택한다.

In [ ]:
# 시간 다양성 필터링 함수 (README 비즈니스 로직 구현)
def time_diversity_filtering(candidates, metadata_df, top_k=3):
    """
    README 5.6절 비즈니스 로직:
    - Top 1: 가장 유사한 패턴
    - Top 2: Top 1과 날짜 차이 최대화
    - Top 3: Top 1, 2와 모두 날짜 차이 최대화
    """
    selected = []
    
    # Top 1: 가장 유사한 패턴
    top1_idx = candidates[0]
    selected.append(top1_idx)
    top1_date = pd.to_datetime(metadata_df.iloc[top1_idx]['start_date'])
    
    if top_k < 2:
        return selected
    
    # Top 2: Top 1과 날짜 차이 최대화
    max_diff = -1
    top2_idx = None
    
    for idx in candidates[1:]:
        date = pd.to_datetime(metadata_df.iloc[idx]['start_date'])
        diff = abs((date - top1_date).days)
        if diff > max_diff:
            max_diff = diff
            top2_idx = idx
    
    if top2_idx is not None:
        selected.append(top2_idx)
    
    if top_k < 3:
        return selected
    
    # Top 3: Top 1, 2와 모두 날짜 차이 최대화
    top2_date = pd.to_datetime(metadata_df.iloc[top2_idx]['start_date'])
    max_min_diff = -1
    top3_idx = None
    
    for idx in candidates[1:]:
        if idx in selected:
            continue
        
        date = pd.to_datetime(metadata_df.iloc[idx]['start_date'])
        diff1 = abs((date - top1_date).days)
        diff2 = abs((date - top2_date).days)
        
        # Top 1, 2와의 최소 거리를 최대화
        min_diff = min(diff1, diff2)
        if min_diff > max_min_diff:
            max_min_diff = min_diff
            top3_idx = idx
    
    if top3_idx is not None:
        selected.append(top3_idx)
    
    return selected


# 검색 함수 정의 (시간 다양성 포함)
def search_similar_patterns_with_diversity(query_idx, k=100, top_k=3):
    """
    query_idx: 쿼리 이미지 인덱스
    k: Faiss에서 가져올 후보 개수 (날짜 다양성을 위해 많이 가져옴)
    top_k: 최종 반환 개수 (기본: 3)
    """
    query_vector = embeddings_matrix[query_idx:query_idx+1]
    distances, indices = index.search(query_vector, k)
    
    # 시간 다양성 필터링 (README 비즈니스 로직)
    selected_indices = time_diversity_filtering(indices[0], metadata_df, top_k=top_k)
    
    # 선택된 인덱스에 해당하는 거리
    selected_distances = [distances[0][list(indices[0]).index(idx)] for idx in selected_indices]
    
    return selected_distances, selected_indices


# 랜덤 쿼리 선택
np.random.seed(42)
query_idx = np.random.randint(0, len(all_paths))

print(f"🔍 쿼리 이미지 인덱스: {query_idx}")
print(f"🔍 쿼리 이미지 경로: {all_paths[query_idx]}")

# 메타데이터 확인
query_meta = metadata_df.iloc[query_idx]
print(f"🔍 쿼리 종목: {query_meta['ticker']}")
print(f"🔍 쿼리 날짜: {query_meta['start_date']}")

# 검색 수행 (시간 다양성 적용)
distances, indices = search_similar_patterns_with_diversity(query_idx, k=100, top_k=3)

print(f"\n✅ Top-3 유사 패턴 (시간 다양성: 날짜 차이 최대화):")
for rank, (idx, dist) in enumerate(zip(indices, distances), 1):
    meta = metadata_df.iloc[idx]
    query_date = pd.to_datetime(query_meta['start_date'])
    result_date = pd.to_datetime(meta['start_date'])
    date_diff_days = abs((result_date - query_date).days)
    date_diff_weeks = date_diff_days / 7
    
    print(f"\n  {rank}. {meta['ticker']} ({meta['start_date']})")
    print(f"     거리: {dist:.4f}, 날짜 차이: {date_diff_weeks:.1f}주 ({date_diff_days}일)")
    print(f"     섹터: {meta['sector']}")
    print(f"     수익률 - 3M: {meta['return_3m']:.1f}%, 6M: {meta['return_6m']:.1f}%, 1Y: {meta['return_1y']:.1f}%")

검색 결과를 시각화하는 함수를 정의하고 단일 쿼리의 Top-3 유사 패턴을 이미지로 출력한다. 티커·날짜·섹터·수익률 메타데이터를 함께 표시해 실전 활용 가치를 확인한다.

In [ ]:
# 검색 결과 시각화 (메타데이터 포함)
def visualize_search_results(query_idx, distances, indices, save_path=None):
    """
    검색 결과를 시각화합니다.
    - 첫 번째 행: 쿼리 이미지
    - 나머지 행: Top-3 유사 패턴 (메타데이터 포함)
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Faiss 검색 결과: Top-3 유사 패턴 (시간 다양성 적용)', fontsize=20, fontweight='bold')
    
    # 첫 번째 행: 쿼리 이미지 (왼쪽 첫 칸)
    query_data = np.load(all_paths[query_idx])
    query_meta = metadata_df.iloc[query_idx]
    
    axes[0, 0].imshow(query_data, cmap='gray', aspect='auto')
    title = f'🔍 쿼리\n{query_meta["ticker"]} ({query_meta["start_date"]})\n섹터: {query_meta["sector"]}'
    axes[0, 0].set_title(title, fontsize=12, fontweight='bold', color='red')
    axes[0, 0].axis('off')
    
    # 나머지 첫 번째 행 빈칸
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    
    # 두 번째 행: Top-3 결과
    for rank, (idx, dist) in enumerate(zip(indices, distances)):
        meta = metadata_df.iloc[idx]
        data = np.load(all_paths[idx])
        
        axes[1, rank].imshow(data, cmap='gray', aspect='auto')
        
        # 메타데이터 표시
        title = (f'Top {rank+1} (유사도: {(1-dist/distances[0]*0.95)*100:.1f}%)\n'
                f'{meta["ticker"]} ({meta["start_date"]})\n'
                f'섹터: {meta["sector"]}\n'
                f'3M: {meta["return_3m"]:.1f}% | 6M: {meta["return_6m"]:.1f}% | 1Y: {meta["return_1y"]:.1f}%')
        
        axes[1, rank].set_title(title, fontsize=10)
        axes[1, rank].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ 시각화 저장: {save_path}")
    
    plt.show()

# 시각화 실행
save_path = OUTPUT_DIR / 'search_results.png'
visualize_search_results(query_idx, distances, indices, save_path=save_path)

## 8. 추가 검색 테스트

3개의 추가 랜덤 쿼리에 대해 Top-3 검색 결과를 시각화한다. 다양한 패턴 유형에 걸쳐 모델의 검색 품질을 정성적으로 확인한다.

3개 추가 쿼리의 검색 결과를 시각화한다.

In [ ]:
# 여러 개의 랜덤 쿼리로 테스트
num_tests = 3
random_indices = np.random.choice(len(all_paths), num_tests, replace=False)

for i, query_idx in enumerate(random_indices, 1):
    print(f"\n{'='*80}")
    print(f"테스트 {i}/{num_tests}: 쿼리 인덱스 {query_idx}")
    query_meta = metadata_df.iloc[query_idx]
    print(f"쿼리: {query_meta['ticker']} ({query_meta['start_date']})")
    print(f"{'='*80}")
    
    distances, indices = search_similar_patterns_with_diversity(query_idx, k=100, top_k=3)
    visualize_search_results(query_idx, distances, indices)

## 9. 성능 평가 및 통계

검색 속도와 거리 분포를 정량적으로 평가한다. 100개 쿼리로 평균 응답 시간을 측정하고, 1,000개 쿼리로 Top-3 거리 분포를 분석한다.

100개 쿼리로 전체 파이프라인(FAISS 검색 + 시간 다양성 필터링)의 평균 응답 시간을 측정한다.

In [ ]:
# 검색 속도 측정
import time

num_queries = 100
query_indices = np.random.choice(len(all_paths), num_queries, replace=False)

start_time = time.time()
for query_idx in query_indices:
    distances, indices = search_similar_patterns_with_diversity(query_idx, k=100, top_k=3)
end_time = time.time()

avg_time = (end_time - start_time) / num_queries

print(f"✅ 평균 검색 시간 (시간 다양성 포함): {avg_time*1000:.2f}ms")
print(f"✅ 초당 검색 수: {1/avg_time:.0f} queries/sec")

1,000개 쿼리의 Top-3 결과 L2 거리 분포를 분석하고 히스토그램으로 저장한다. 평균 거리와 분포를 보면 모델이 다양한 쿼리에서 일관된 유사도 수준을 유지하는지 파악할 수 있다.

In [ ]:
# 거리 분포 분석 (시간 다양성 적용)
sample_size = 1000
sample_indices = np.random.choice(len(all_paths), sample_size, replace=False)

all_distances = []
for query_idx in tqdm(sample_indices, desc="거리 분포 계산 중"):
    distances, _ = search_similar_patterns_with_diversity(query_idx, k=100, top_k=3)
    all_distances.extend(distances[1:])  # 자기 자신 제외

all_distances = np.array(all_distances)

print(f"\n📊 Top-3 거리 통계 (시간 다양성 적용):")
print(f"  평균: {all_distances.mean():.4f}")
print(f"  중앙값: {np.median(all_distances):.4f}")
print(f"  표준편차: {all_distances.std():.4f}")
print(f"  최솟값: {all_distances.min():.4f}")
print(f"  최댓값: {all_distances.max():.4f}")

# 히스토그램
plt.figure(figsize=(10, 6))
plt.hist(all_distances, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('L2 Distance', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Top-3 검색 결과 거리 분포 (시간 다양성 적용)', fontsize=14, fontweight='bold')
plt.axvline(all_distances.mean(), color='red', linestyle='--', label=f'평균: {all_distances.mean():.4f}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distance_distribution.png', dpi=150)
plt.show()

print(f"\n✅ 거리 분포 그래프 저장: {OUTPUT_DIR / 'distance_distribution.png'}")

## 10. 모델 학습 품질 평가

50개 종목 × 10개 패턴을 쿼리로 사용해 FAISS 검색 결과의 품질을 정량 평가한다.

핵심 지표: Top-10 중 **같은 종목 패턴이 몇 개 포함되는가**. 이상적으로는 다양한 종목에서 유사 패턴을 찾아야 하므로, 같은 종목 수가 낮을수록 좋다. 동시에 같은 종목 패턴의 L2 거리가 다른 종목보다 가깝다면, 모델이 차트 모양을 잘 임베딩하고 있다는 증거다.

전체 172개 종목 중 50개를 시드 42로 무작위 선택한다. 다양한 섹터와 종목이 포함되도록 랜덤 샘플링을 사용한다.

In [ ]:
# 1. 50개 종목 랜덤 선택
np.random.seed(42)
all_tickers = metadata_df['ticker'].unique()
selected_tickers = np.random.choice(all_tickers, 50, replace=False)

print(f"📊 선택된 50개 종목:")
print(f"{sorted(selected_tickers)}")
print(f"\n✅ 총 {len(selected_tickers)}개 종목 선택 완료")

선택된 50개 종목 각각에서 최대 10개 패턴을 무작위 샘플링해 쿼리 목록을 생성한다. 총 500개 쿼리가 평가에 사용된다.

In [ ]:
# 2. 각 종목에서 10개 패턴 랜덤 선택
query_list = []

for ticker in selected_tickers:
    ticker_patterns = metadata_df[metadata_df['ticker'] == ticker]
    
    # 10개보다 적으면 있는 만큼만
    n_samples = min(10, len(ticker_patterns))
    sampled = ticker_patterns.sample(n=n_samples, random_state=42)
    
    for idx in sampled.index:
        query_list.append({
            'query_idx': idx,
            'query_ticker': ticker
        })

print(f"✅ 총 쿼리 개수: {len(query_list)}개")
print(f"✅ 평균 종목당 패턴: {len(query_list) / len(selected_tickers):.1f}개")

500개 쿼리 각각에 대해 FAISS Top-11을 검색하고 자기 자신을 제외해 Top-10을 평가한다. 같은 종목과 다른 종목의 결과를 분리해 평균 거리와 최솟값을 기록한다.

In [ ]:
# 3. 검색 및 평가
results = []

print("\n🔍 검색 및 평가 시작...\n")

for query_info in tqdm(query_list, desc="평가 진행"):
    query_idx = query_info['query_idx']
    query_ticker = query_info['query_ticker']
    
    # Faiss 검색 (자기 자신 포함 11개 검색)
    query_vector = embeddings_matrix[query_idx:query_idx+1]
    distances, indices = index.search(query_vector, 11)
    
    # 자기 자신 제외
    distances = distances[0][1:11]  # Top-10
    indices = indices[0][1:11]
    
    # Top-10 중 같은 티커 찾기
    same_ticker_indices = []
    same_ticker_distances = []
    diff_ticker_indices = []
    diff_ticker_distances = []
    
    for idx, dist in zip(indices, distances):
        result_ticker = metadata_df.iloc[idx]['ticker']
        
        if result_ticker == query_ticker:
            same_ticker_indices.append(idx)
            same_ticker_distances.append(dist)
        else:
            diff_ticker_indices.append(idx)
            diff_ticker_distances.append(dist)
    
    # 통계 계산
    same_count = len(same_ticker_indices)
    diff_count = len(diff_ticker_indices)
    
    avg_same_dist = np.mean(same_ticker_distances) if same_count > 0 else np.nan
    avg_diff_dist = np.mean(diff_ticker_distances) if diff_count > 0 else np.nan
    avg_all_dist = np.mean(distances)
    
    min_same_dist = np.min(same_ticker_distances) if same_count > 0 else np.nan
    min_diff_dist = np.min(diff_ticker_distances) if diff_count > 0 else np.nan
    
    # 결과 저장
    results.append({
        'query_ticker': query_ticker,
        'query_idx': query_idx,
        'same_ticker_count': same_count,
        'diff_ticker_count': diff_count,
        'avg_same_dist': avg_same_dist,
        'avg_diff_dist': avg_diff_dist,
        'avg_all_dist': avg_all_dist,
        'min_same_dist': min_same_dist,
        'min_diff_dist': min_diff_dist,
        'same_ticker_distances': same_ticker_distances,
        'diff_ticker_distances': diff_ticker_distances
    })

# DataFrame으로 변환
results_df = pd.DataFrame(results)

print("\n✅ 평가 완료!")
print(f"✅ 총 {len(results_df)}개 쿼리 평가됨")

Top-10 중 같은 종목 개수 분포를 출력한다. 평균이 낮을수록 모델이 종목 편향 없이 형태가 유사한 패턴을 찾고 있다는 의미다.

In [ ]:
# 4. 결과 분석
print("="*80)
print("📊 모델 학습 품질 평가 결과")
print("="*80)

# 1. 같은 티커 개수 분포
print(f"\n【1】 Top-10 중 같은 티커 개수 분포")
print("-" * 80)

same_ticker_counts = results_df['same_ticker_count'].value_counts().sort_index()
print(f"\n총 쿼리 수: {len(results_df)}개\n")

for count in range(11):
    freq = same_ticker_counts.get(count, 0)
    pct = freq / len(results_df) * 100
    bar = "█" * int(pct / 2)
    print(f"{count:2d}개: {freq:4d}회 ({pct:5.1f}%) {bar}")

avg_same = results_df['same_ticker_count'].mean()
median_same = results_df['same_ticker_count'].median()
print(f"\n평균: {avg_same:.2f}개")
print(f"중앙값: {median_same:.0f}개")

같은 종목과 다른 종목의 L2 거리를 따로 계산한다. 두 그룹의 평균 거리 차이(gap)가 클수록 모델이 패턴 유사도를 잘 구분하고 있다는 의미다.

In [ ]:
# 2. 거리 통계
print(f"\n【2】 거리 통계")
print("-" * 80)

# 전체 평균
print(f"\n전체 Top-10 평균 거리: {results_df['avg_all_dist'].mean():.4f}")

# 같은 티커 vs 다른 티커
valid_same = results_df[results_df['same_ticker_count'] > 0]
valid_diff = results_df[results_df['diff_ticker_count'] > 0]

print(f"\n같은 티커 평균 거리: {valid_same['avg_same_dist'].mean():.4f}")
print(f"  - 최솟값: {valid_same['min_same_dist'].min():.4f}")
print(f"  - 최댓값: {valid_same['avg_same_dist'].max():.4f}")
print(f"  - 표준편차: {valid_same['avg_same_dist'].std():.4f}")

print(f"\n다른 티커 평균 거리: {valid_diff['avg_diff_dist'].mean():.4f}")
print(f"  - 최솟값: {valid_diff['min_diff_dist'].min():.4f}")
print(f"  - 최댓값: {valid_diff['avg_diff_dist'].max():.4f}")
print(f"  - 표준편차: {valid_diff['avg_diff_dist'].std():.4f}")

# 거리 차이
both_valid = results_df[(results_df['same_ticker_count'] > 0) & (results_df['diff_ticker_count'] > 0)]
distance_gap = (both_valid['avg_diff_dist'] - both_valid['avg_same_dist']).mean()
print(f"\n⭐ 평균 거리 차이 (다른티커 - 같은티커): {distance_gap:.4f}")

같은 종목 개수별로 쿼리를 그룹화하고 각 그룹의 평균 거리를 비교한다. 종목 편향이 심한 케이스(같은 종목 많음)와 다양한 케이스(같은 종목 적음)의 거리 패턴을 분석한다.

In [ ]:
# 3. 케이스별 상세 분석
print(f"\n【3】 케이스별 상세 분석")
print("-" * 80)

# 같은 티커 개수별로 그룹화
for same_count in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    group = results_df[results_df['same_ticker_count'] == same_count]
    
    if len(group) == 0:
        continue
    
    print(f"\n같은 티커 {same_count}개인 경우 ({len(group)}회):")
    print(f"  전체 평균 거리: {group['avg_all_dist'].mean():.4f}")
    
    if same_count > 0:
        print(f"  같은 티커 평균 거리: {group['avg_same_dist'].mean():.4f}")
    if same_count < 10:
        print(f"  다른 티커 평균 거리: {group['avg_diff_dist'].mean():.4f}")

Top-1이 다른 종목인 비율, Top-10 중 다른 종목 3개 이상 비율 등 종합 품질 지표를 계산한다. 실험 1 모델의 종목 편향도와 Positive/Negative 구분력을 최종 판정한다.

In [ ]:
# 4. 종합 평가
print(f"\n【4】 종합 평가")
print("-" * 80)

# 평가 지표
top1_diff_ticker = len(results_df[results_df['same_ticker_count'] == 0])
top1_diff_ticker_pct = top1_diff_ticker / len(results_df) * 100

top3_has_diff = len(results_df[results_df['diff_ticker_count'] >= 3])
top3_has_diff_pct = top3_has_diff / len(results_df) * 100

print(f"\nTop-1이 다른 티커: {top1_diff_ticker}회 ({top1_diff_ticker_pct:.1f}%)")
print(f"Top-10 중 다른 티커 3개 이상: {top3_has_diff}회 ({top3_has_diff_pct:.1f}%)")

# 평가
print(f"\n📋 평가 기준:")
if avg_same < 3:
    ticker_bias = "✅ 우수 (낮은 종목 편향)"
elif avg_same < 5:
    ticker_bias = "⚠️ 보통 (약간의 종목 편향)"
else:
    ticker_bias = "❌ 나쁨 (높은 종목 편향)"

print(f"  종목 편향도: {ticker_bias} (평균 {avg_same:.2f}개)")

if distance_gap > 0.05:
    separation = "✅ 우수 (Positive/Negative 구분 명확)"
elif distance_gap > 0.02:
    separation = "⚠️ 보통 (Positive/Negative 구분 보통)"
else:
    separation = "❌ 나쁨 (Positive/Negative 구분 약함)"

print(f"  구분력: {separation} (거리 차이 {distance_gap:.4f})")

print("\n" + "="*80)

같은 종목 개수 분포, 거리 히스토그램, 같은/다른 종목 거리 산점도 등 2×3 그리드 시각화를 생성하고 저장한다.

In [ ]:
# 5. 시각화
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('모델 학습 품질 평가 결과', fontsize=16, fontweight='bold')

# 1. 같은 티커 개수 분포
same_counts = results_df['same_ticker_count'].values
axes[0, 0].hist(same_counts, bins=range(12), edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(avg_same, color='red', linestyle='--', 
                   label=f'평균: {avg_same:.2f}', linewidth=2)
axes[0, 0].set_xlabel('Top-10 중 같은 티커 개수', fontsize=12)
axes[0, 0].set_ylabel('빈도', fontsize=12)
axes[0, 0].set_title('같은 티커 개수 분포', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. 전체 평균 거리 분포
axes[0, 1].hist(results_df['avg_all_dist'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].axvline(results_df['avg_all_dist'].mean(), color='red', linestyle='--',
                   label=f'평균: {results_df["avg_all_dist"].mean():.4f}', linewidth=2)
axes[0, 1].set_xlabel('평균 거리', fontsize=12)
axes[0, 1].set_ylabel('빈도', fontsize=12)
axes[0, 1].set_title('Top-10 평균 거리 분포', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. 같은 티커 vs 다른 티커 거리 비교
both_valid_plot = results_df[(results_df['same_ticker_count'] > 0) & (results_df['diff_ticker_count'] > 0)]
axes[0, 2].scatter(both_valid_plot['avg_same_dist'], both_valid_plot['avg_diff_dist'], alpha=0.5)
axes[0, 2].plot([0, 0.5], [0, 0.5], 'r--', label='y=x', linewidth=2)
axes[0, 2].set_xlabel('같은 티커 평균 거리', fontsize=12)
axes[0, 2].set_ylabel('다른 티커 평균 거리', fontsize=12)
axes[0, 2].set_title('같은 티커 vs 다른 티커 거리', fontsize=13, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# 4. 같은 티커 개수별 평균 거리
same_count_groups = results_df.groupby('same_ticker_count')['avg_all_dist'].mean()
axes[1, 0].bar(same_count_groups.index, same_count_groups.values, 
               edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('같은 티커 개수', fontsize=12)
axes[1, 0].set_ylabel('평균 거리', fontsize=12)
axes[1, 0].set_title('같은 티커 개수별 평균 거리', fontsize=13, fontweight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# 5. 같은 티커 거리 분포 (있는 경우만)
same_distances = []
for dists in valid_same['same_ticker_distances']:
    same_distances.extend(dists)
axes[1, 1].hist(same_distances, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].axvline(np.mean(same_distances), color='red', linestyle='--',
                   label=f'평균: {np.mean(same_distances):.4f}', linewidth=2)
axes[1, 1].set_xlabel('거리', fontsize=12)
axes[1, 1].set_ylabel('빈도', fontsize=12)
axes[1, 1].set_title('같은 티커 거리 분포', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

# 6. 다른 티커 거리 분포
diff_distances = []
for dists in valid_diff['diff_ticker_distances']:
    diff_distances.extend(dists)
axes[1, 2].hist(diff_distances, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 2].axvline(np.mean(diff_distances), color='red', linestyle='--',
                   label=f'평균: {np.mean(diff_distances):.4f}', linewidth=2)
axes[1, 2].set_xlabel('거리', fontsize=12)
axes[1, 2].set_ylabel('빈도', fontsize=12)
axes[1, 2].set_title('다른 티커 거리 분포', fontsize=13, fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ 시각화 저장: {OUTPUT_DIR / 'model_evaluation.png'}")

평가 결과를 CSV 파일로 저장한다. 쿼리별 상세 결과와 요약 통계를 Google Drive에 보관해 07 노트북의 실험 간 비교에 활용한다.

In [ ]:
# 6. 결과 저장
# CSV 저장 (거리 리스트는 제외)
results_save = results_df.drop(columns=['same_ticker_distances', 'diff_ticker_distances'])
results_save.to_csv(OUTPUT_DIR / 'evaluation_results.csv', index=False)
print(f"✅ 결과 저장: {OUTPUT_DIR / 'evaluation_results.csv'}")

# 요약 통계 저장
summary_stats = {
    'total_queries': len(results_df),
    'avg_same_ticker_count': avg_same,
    'median_same_ticker_count': median_same,
    'avg_all_distance': results_df['avg_all_dist'].mean(),
    'avg_same_ticker_distance': valid_same['avg_same_dist'].mean(),
    'avg_diff_ticker_distance': valid_diff['avg_diff_dist'].mean(),
    'distance_gap': distance_gap,
    'top1_diff_ticker_count': top1_diff_ticker,
    'top1_diff_ticker_pct': top1_diff_ticker_pct
}

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv(OUTPUT_DIR / 'evaluation_summary.csv', index=False)
print(f"✅ 요약 통계 저장: {OUTPUT_DIR / 'evaluation_summary.csv'}")

print("\n" + "="*80)
print("🎉 모델 평가 완료!")
print("="*80)
print(f"\n저장된 파일:")
print(f"  - {OUTPUT_DIR / 'evaluation_results.csv'}")
print(f"  - {OUTPUT_DIR / 'evaluation_summary.csv'}")
print(f"  - {OUTPUT_DIR / 'model_evaluation.png'}")
print("="*80)

## 🎯 완료 체크리스트

### ✅ 완료된 작업:
1. ✅ ResNet18 모델 로드 및 검증
2. ✅ 49,987개 이미지 임베딩 생성 (512차원)
3. ✅ Faiss 인덱스 구축 (L2 거리)
4. ✅ Top-10 검색 기능 구현
5. ✅ 검색 결과 시각화 (기존 이미지를 쿼리로 사용)
6. ✅ 성능 측정 및 통계 분석

### 💾 저장된 파일:
- `/content/drive/MyDrive/Patron/faiss/embeddings.npy` (임베딩 벡터)
- `/content/drive/MyDrive/Patron/faiss/image_paths.pkl` (이미지 경로)
- `/content/drive/MyDrive/Patron/faiss/faiss_index.bin` (Faiss 인덱스)
- `/content/drive/MyDrive/Patron/faiss/search_results.png` (검색 결과)
- `/content/drive/MyDrive/Patron/faiss/distance_distribution.png` (거리 분포)

### 🚀 다음 단계:
1. **FastAPI 서버 구축** (patron_06_fastapi.py)
   - 새로운 차트 업로드 기능
   - 검색 API 엔드포인트
   - 실시간 차트 유사도 검색
2. **프론트엔드 연동** (React)
   - 차트 업로드 인터페이스
   - 검색 결과 시각화
3. **모델 최적화** (patron_04_올인원_최종.ipynb)
   - 하이퍼파라미터 튜닝
   - Loss 함수 개선